In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv(override=True)

True

In [ ]:
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if google_api_key:
    print("Gemini")
if groq_api_key:
    print("groq_api_key")

Gemini
groq_api_key


In [18]:
from IPython.display import display, Markdown

gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
def generate_code(task):
    model_name = "gemini-flash-latest"
    return  gemini.chat.completions.create(
                model= model_name,
                messages=[{
                    "role" : "user",
                    "content": task
                }]
            ).choices[0].message.content


def critique_code(code):
    model_name = "openai/gpt-oss-120b"
    return groq.chat.completions.create(
        model=model_name,
        messages=[{
                    "role" : "user",
                    "content": f"Critique this code and suggest improvements:\n{code},don't write code,just feedback is enough"
                }]
    ).choices[0].message.content


def refine_code(code, feedback):
    model_name = "gemini-flash-latest"
    return gemini.chat.completions.create(
                model= model_name,
                messages=[{
                    "role" : "user",
                    "content": f"Improve this code:\n{code}\nFeedback:\n{feedback}"
                }]
            ).choices[0].message.content

task = "Write an efficient palindrome checker in Python"
code = generate_code(task)
feedback = critique_code(code)
final_code = refine_code(code,feedback)

display(Markdown(final_code))

Based on the excellent feedback provided, I have refined the implementations. These versions now include **type hinting**, **enhanced Unicode support**, **optimized logic**, and **clear documentation** regarding the trade-offs of each approach.

---

### 1. The "Pythonic" Way (Optimized Slicing)
This is the standard choice for production. We switched `lower()` to `casefold()` to better handle international characters (like the German "ß") and added a type guard.

*   **Time Complexity:** $O(n)$
*   **Space Complexity:** $O(n)$
*   **Best for:** General purpose, readability, and speed.

```python
def is_palindrome_slicing(s: str) -> bool:
    """
    Checks if a string is a palindrome using Python's optimized slicing.
    Handles Unicode characters and ignores casing/punctuation.
    """
    if not isinstance(s, str):
        raise TypeError(f"Expected string, got {type(s).__name__}")

    # .casefold() is more aggressive than .lower() for international strings
    clean_s = "".join(char.casefold() for char in s if char.isalnum())
    return clean_s == clean_s[::-1]

# Example:
print(is_palindrome_slicing("A man, a plan, a canal: Panama"))  # True
```

---

### 2. The Memory-Efficient Way (Iterative Two-Pointer)
This version avoids creating any new strings (until the comparison step), making it ideal for extremely large inputs where memory is a constraint.

*   **Time Complexity:** $O(n)$
*   **Space Complexity:** $O(1)$
*   **Best for:** Technical interviews and processing massive text files.

```python
def is_palindrome_iterative(s: str) -> bool:
    """
    Checks if a string is a palindrome using two pointers.
    Memory efficient: O(1) extra space.
    """
    if not isinstance(s, str):
        raise TypeError(f"Expected string, got {type(s).__name__}")

    left, right = 0, len(s) - 1
    
    while left < right:
        # Advance pointers if characters are not alphanumeric
        if not s[left].isalnum():
            left += 1
        elif not s[right].isalnum():
            right -= 1
        else:
            # Compare using casefold for better Unicode support
            if s[left].casefold() != s[right].casefold():
                return False
            left += 1
            right -= 1
            
    return True
```

---

### 3. The "Pure Logic" Way (Recursive)
**Note:** This is provided for academic interest. In Python, this is inefficient because string slicing `s[1:-1]` creates a new copy of the string every time, leading to $O(n^2)$ complexity. It will also hit the `RecursionError` on strings longer than ~1000 characters.

```python
def is_palindrome_recursive(s: str) -> bool:
    """
    Educational implementation using recursion.
    Warning: Inefficient for large strings in Python.
    """
    # Base case: empty or single char is a palindrome
    if len(s) <= 1:
        return True
    
    # Recursive step: logic only works on pre-cleaned strings
    # To keep this example simple, assume s is already cleaned/lowered
    if s[0] != s[-1]:
        return False
    return is_palindrome_recursive(s[1:-1])
```

---

### Which one should you use?

| Method | Best For | Why? |
| :--- | :--- | :--- |
| **Slicing** | **Production** | It is written in C. Even though it creates a copy, it is usually the fastest in real-world benchmarks for standard strings. |
| **Two-Pointer** | **Memory/Interviews** | It proves you understand how to traverse data without allocating extra memory. It handles multi-GB strings safely. |
| **Recursive** | **Learning** | Good for understanding the base-case/recursive-step pattern, but dangerous for production. |

### Advanced Tip: Unicode Normalization
If you are dealing with complex Unicode (e.g., characters with combining accents), a string might look the same but have different byte representations. To fix this, import `unicodedata` and normalize your string before checking:

```python
import unicodedata

def normalize_string(s: str) -> str:
    # Converts combined characters into a standard form (NFC)
    return unicodedata.normalize('NFC', s)
```

### Quick Test Suite
You can use this block to verify all methods at once:

```python
test_cases = [
    ("Racecar", True),
    ("A man, a plan, a canal: Panama", True),
    ("Hello", False),
    (".,.", True),
    ("12321", True),
    ("German: Straße", False) # 'ße' != 'eß'
]

for text, expected in test_cases:
    assert is_palindrome_slicing(text) == expected
    assert is_palindrome_iterative(text) == expected
print("All tests passed!")
```